In [ ]:
import zipfile
import json
import pandas as pd

zip_path = r'C:\Users\felip\Downloads\archive.zip' #https://www.kaggle.com/datasets/johnarevalo/raw-mmimdb

# Fix during loading - change plots[0] to plots[-1] in flatten_record
def flatten_record(record, movie_id):
    flat = {}
    flat['movie_id'] = movie_id
    
    scalar_fields = [
        'title', 'canonical title', 'smart canonical title',
        'long imdb title', 'long imdb canonical title',
        'smart long imdb canonical title', 'year', 'rating',
        'votes', 'kind', 'plot outline', 'aspect ratio',
        'cover url', 'full-size cover url'
    ]
    for field in scalar_fields:
        flat[field] = record.get(field, None)
    
    list_fields = [
        'runtimes', 'color info', 'certificates', 'country codes',
        'countries', 'genres', 'sound mix', 'akas'
    ]
    for field in list_fields:
        val = record.get(field, [])
        flat[field] = ' | '.join(val) if val else None
    
    plots = record.get('plot', [])
    flat['plot'] = plots[-1] if plots else None  # ← changed to -1

    for person_field in ['director', 'cast', 'cinematographer']:
        val = record.get(person_field, [])
        flat[person_field] = ' | '.join([p.get('name', '') for p in val]) if val else None

    for company_field in ['distributors', 'production companies']:
        val = record.get(company_field, [])
        flat[company_field] = ' | '.join([c.get('name', '') for c in val]) if val else None

    return flat

# First let's peek inside the zip to find the right path
print("Scanning zip structure...")
with zipfile.ZipFile(zip_path, 'r') as z:
    all_files = z.namelist()
    # Show first 10 files to confirm structure
    print("First 10 files:", all_files[:10])
    
    # Get all json files
    json_files = [f for f in all_files if f.endswith('.json')]
    print(f"\nFound {len(json_files)} JSON files")
    
    records = []
    for i, file in enumerate(json_files):
        try:
            with z.open(file) as f:
                record = json.load(f)
                movie_id = file.split('/')[-1].replace('.json', '')
                records.append(flatten_record(record, movie_id))
        except Exception as e:
            print(f"Skipped {file}: {e}")
        
        if i % 1000 == 0:
            print(f"Progress: {i}/{len(json_files)}...")

df = pd.DataFrame(records)

Scanning zip structure...
First 10 files: ['mmimdb/dataset/0000005.jpeg', 'mmimdb/dataset/0000005.json', 'mmimdb/dataset/0000008.jpeg', 'mmimdb/dataset/0000008.json', 'mmimdb/dataset/0000010.jpeg', 'mmimdb/dataset/0000010.json', 'mmimdb/dataset/0000012.jpeg', 'mmimdb/dataset/0000012.json', 'mmimdb/dataset/0000014.jpeg', 'mmimdb/dataset/0000014.json']

Found 51920 JSON files
Progress: 0/51920...
Progress: 1000/51920...
Progress: 2000/51920...
Progress: 3000/51920...
Progress: 4000/51920...
Progress: 5000/51920...
Progress: 6000/51920...
Progress: 7000/51920...
Progress: 8000/51920...
Progress: 9000/51920...
Progress: 10000/51920...
Progress: 11000/51920...
Progress: 12000/51920...
Progress: 13000/51920...
Progress: 14000/51920...
Progress: 15000/51920...
Progress: 16000/51920...
Progress: 17000/51920...
Progress: 18000/51920...
Progress: 19000/51920...
Progress: 20000/51920...
Progress: 21000/51920...
Progress: 22000/51920...
Progress: 23000/51920...
Progress: 24000/51920...
Progress: 2

In [17]:
df['plot'] = df['plot'].str.lower()
df['genres'] = df['genres'].apply(lambda x: [g.strip() for g in x.split('|')] if pd.notna(x) else [])
df['genres'] = df['genres'].astype(str)
df[['year','title','plot','genres','rating']]

,year,title,plot,genres,rating
0,1893.0,Blacksmith Scene,three men hammer on an anvil and pass a bottle...,['Short'],6.2
1,1894.0,Edison Kinetoscopic Record of a Sneeze,a man (,"['Documentary', 'Short']",5.7
2,1895.0,La Sortie de l'Usine Lumière à Lyon (le Premie...,a man opens the big gates to the lumière facto...,"['Documentary', 'Short']",6.9
3,1896.0,L'arrivée d'un train à La Ciotat,a group of people are standing in a straight l...,"['Documentary', 'Short']",7.4
4,1895.0,L'arroseur arrosé,"a gardener is watering his flowers, when a mis...","['Comedy', 'Short']",7.2
...,...,...,...,...,...
51915,2014.0,Polskie gówno,indie rock band tranzystory goes on its first ...,"['Comedy', 'Musical']",5.4
51916,2015.0,Power/Rangers,a dark and gritty re-imagining of the classic ...,"['Short', 'Action', 'Sci-Fi', 'Thriller']",7.9
51917,2015.0,Aziz Ansari Live in Madison Square Garden,stand-up comedian and tv star aziz ansari deli...,['Comedy'],6.7
51918,NaN,NaN,NaN,[],NaN


In [24]:
df.columns

Index(['movie_id', 'title', 'canonical title', 'smart canonical title',
       'long imdb title', 'long imdb canonical title',
       'smart long imdb canonical title', 'year', 'rating', 'votes', 'kind',
       'plot outline', 'aspect ratio', 'cover url', 'full-size cover url',
       'runtimes', 'color info', 'certificates', 'country codes', 'countries',
       'genres', 'sound mix', 'akas', 'plot', 'director', 'cast',
       'cinematographer', 'distributors', 'production companies'],
      dtype='str')

In [3]:
dataTraining = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTraining.zip', encoding='UTF-8', index_col=0)

In [18]:
mer=dataTraining.merge(df[['year','title','plot','plot outline','genres','rating']],on=['title','year','rating'],how='left',indicator=True)

In [10]:
len(mer)

15794

In [11]:
mer

,year,title,plot_x,genres_x,rating,plot_y,plot outline,genres_y,_merge
0,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0,a poetic and powerful story of a father forced...,A poetic and powerful story of a father forced...,"[Short, Drama]",both
1,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0,a poetic and powerful story of a father forced...,A poetic and powerful story of a father forced...,"[Short, Drama]",both
2,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6,a serial killer decides to teach the secrets o...,A serial killer decides to teach the secrets o...,"[Comedy, Crime, Horror]",both
3,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6,a serial killer decides to teach the secrets o...,A serial killer decides to teach the secrets o...,"[Comedy, Crime, Horror]",both
4,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2,a female blackmailer with a disfiguring facial...,A female blackmailer with a disfiguring facial...,"[Drama, Film-Noir, Thriller]",both
...,...,...,...,...,...,...,...,...,...
15789,1955,Kismet,"like a tale spun by scheherazade , kismet fol...","['Adventure', 'Musical', 'Fantasy', 'Comedy', ...",6.4,a roguish poet is given the run of the schemin...,A roguish poet is given the run of the schemin...,"[Adventure, Musical, Fantasy, Comedy, Romance]",both
15790,1982,The Secret of NIMH,"mrs . brisby , a widowed mouse , lives in a...","['Animation', 'Adventure', 'Drama', 'Family', ...",7.6,"to save her ill son, a field mouse must seek t...","To save her ill son, a field mouse must seek t...","[Animation, Adventure, Drama, Family, Fantasy,...",both
15791,1982,The Secret of NIMH,"mrs . brisby , a widowed mouse , lives in a...","['Animation', 'Adventure', 'Drama', 'Family', ...",7.6,"to save her ill son, a field mouse must seek t...","To save her ill son, a field mouse must seek t...","[Animation, Adventure, Drama, Family, Fantasy,...",both
15792,2009,Tinker Bell and the Lost Treasure,tinker bell journey far north of never land to...,"['Animation', 'Adventure', 'Family', 'Fantasy']",6.8,tinker bell journey far north of never land to...,Tinker Bell journey far North of Never Land to...,"[Animation, Adventure, Family, Fantasy]",both


In [19]:
mer[mer['genres_x']==mer['genres_y']]

,year,title,plot_x,genres_x,rating,plot_y,plot outline,genres_y,_merge
0,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0,a poetic and powerful story of a father forced...,A poetic and powerful story of a father forced...,"['Short', 'Drama']",both
1,2003,Most,most is the story of a single father who takes...,"['Short', 'Drama']",8.0,a poetic and powerful story of a father forced...,A poetic and powerful story of a father forced...,"['Short', 'Drama']",both
2,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6,a serial killer decides to teach the secrets o...,A serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",both
3,2008,How to Be a Serial Killer,a serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",5.6,a serial killer decides to teach the secrets o...,A serial killer decides to teach the secrets o...,"['Comedy', 'Crime', 'Horror']",both
4,1941,A Woman's Face,"in sweden , a female blackmailer with a disfi...","['Drama', 'Film-Noir', 'Thriller']",7.2,a female blackmailer with a disfiguring facial...,A female blackmailer with a disfiguring facial...,"['Drama', 'Film-Noir', 'Thriller']",both
...,...,...,...,...,...,...,...,...,...
15789,1955,Kismet,"like a tale spun by scheherazade , kismet fol...","['Adventure', 'Musical', 'Fantasy', 'Comedy', ...",6.4,a roguish poet is given the run of the schemin...,A roguish poet is given the run of the schemin...,"['Adventure', 'Musical', 'Fantasy', 'Comedy', ...",both
15790,1982,The Secret of NIMH,"mrs . brisby , a widowed mouse , lives in a...","['Animation', 'Adventure', 'Drama', 'Family', ...",7.6,"to save her ill son, a field mouse must seek t...","To save her ill son, a field mouse must seek t...","['Animation', 'Adventure', 'Drama', 'Family', ...",both
15791,1982,The Secret of NIMH,"mrs . brisby , a widowed mouse , lives in a...","['Animation', 'Adventure', 'Drama', 'Family', ...",7.6,"to save her ill son, a field mouse must seek t...","To save her ill son, a field mouse must seek t...","['Animation', 'Adventure', 'Drama', 'Family', ...",both
15792,2009,Tinker Bell and the Lost Treasure,tinker bell journey far north of never land to...,"['Animation', 'Adventure', 'Family', 'Fantasy']",6.8,tinker bell journey far north of never land to...,Tinker Bell journey far North of Never Land to...,"['Animation', 'Adventure', 'Family', 'Fantasy']",both


In [43]:
dataTesting = pd.read_csv('https://github.com/albahnsen/MIAD_ML_and_NLP/raw/main/datasets/dataTesting.zip', encoding='UTF-8', index_col=0)
dataTesting=dataTesting.reset_index(names='ID')
dataTesting

,ID,year,title,plot
0,1,1999,Message in a Bottle,"who meets by fate , shall be sealed by fate ...."
1,4,1978,Midnight Express,"the true story of billy hayes , an american c..."
2,5,1996,Primal Fear,martin vail left the chicago da ' s office to ...
3,6,1950,Crisis,husband and wife americans dr . eugene and mr...
4,7,1959,The Tingler,the coroner and scientist dr . warren chapin ...
...,...,...,...,...
3378,11263,2008,The Fifth Commandment,"in bangkok , an assassin who turns down a job..."
3379,11265,2003,Coffee and Cigarettes,eleven separate vignettes are presented . in ...
3380,11269,1957,Pal Joey,"joey evans is charming , handsome , funny , ..."
3381,11270,2002,Jonah: A VeggieTales Movie,when the singing veggies encounter some car tr...


In [48]:
df_final=dataTesting.merge(df[['year','title','plot','plot outline','genres','rating']],on=['title','year'],how='left',indicator=True)
df_final['genres'] = df_final['genres'].apply(
    lambda x: tuple(x) if isinstance(x, list) else x
)

df_final = df_final.drop_duplicates(
    subset=['title', 'year', 'genres']
)

df_final = df_final.drop_duplicates(subset=['title','year'])
df_final

,ID,year,title,plot_x,plot_y,plot outline,genres,rating,_merge
0,1,1999,Message in a Bottle,"who meets by fate , shall be sealed by fate ....",a woman discovers a tragic love letter in a bo...,A woman discovers a tragic love letter in a bo...,"(Drama, Romance)",6.1,both
2,4,1978,Midnight Express,"the true story of billy hayes , an american c...","the true story of billy hayes, an american col...","The true story of Billy Hayes, an American col...","(Biography, Crime, Drama)",7.7,both
4,5,1996,Primal Fear,martin vail left the chicago da ' s office to ...,"an altar boy is accused of murdering a priest,...","An altar boy is accused of murdering a priest,...","(Crime, Drama, Mystery, Thriller)",7.7,both
6,6,1950,Crisis,husband and wife americans dr . eugene and mr...,"dr ferguson is a brain surgeon, on vacation wi...",Husband and wife Americans Dr. Eugene and Mrs....,"(Crime, Drama, Thriller)",6.7,both
8,7,1959,The Tingler,the coroner and scientist dr . warren chapin ...,the coroner and scientist dr. warren chapin is...,Dr. Warren Chapin is a pathologist who regular...,"(Horror,)",6.7,both
...,...,...,...,...,...,...,...,...,...
6758,11263,2008,The Fifth Commandment,"in bangkok , an assassin who turns down a job...","in bangkok, an assassin who turns down a job t...","In Bangkok, an assassin who turns down a job t...","(Action, Comedy, Crime, Drama, Thriller)",4.6,both
6760,11265,2003,Coffee and Cigarettes,eleven separate vignettes are presented . in ...,a series of vignettes that all have coffee and...,A series of vignettes that all have coffee and...,"(Comedy, Drama, Music)",7.1,both
6762,11269,1957,Pal Joey,"joey evans is charming , handsome , funny , ...","joey evans is charming, handsome, funny, talen...","Joey Evans is charming, handsome, funny, talen...","(Drama, Musical, Romance)",6.8,both
6764,11270,2002,Jonah: A VeggieTales Movie,when the singing veggies encounter some car tr...,when the singing veggies encounter some car tr...,When the singing Veggies encounter some car tr...,"(Animation, Adventure, Comedy, Drama, Family, ...",6.6,both


In [52]:
import pandas as pd
import ast

# Load sample submission to get exact column order
sample = pd.read_csv(r"C:\Users\felip\Downloads\test-miad-2024-12\sample_submission.csv")

# Genre columns in the exact order Kaggle expects
genre_cols = [col for col in sample.columns if col.startswith('p_')]
genres_order = [col.replace('p_', '') for col in genre_cols]
print("Genres order:", genres_order)

# Make sure genres column is a proper list
df_final['genres'] = df_final['genres'].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)

# One-hot encode in the exact same order as sample submission
for genre in genres_order:
    df_final[f'p_{genre}'] = df_final['genres'].apply(lambda x: 1 if genre in x else 0)

# Build submission dataframe
submission = df_final[['ID'] + genre_cols].copy()

# Verify shape matches sample
print(f"Submission shape: {submission.shape}")
print(f"Sample shape:     {sample.shape}")

# Save
submission.to_csv(r'C:\Users\felip\Downloads\solution_file.csv', index=False)

Genres order: ['Action', 'Adventure', 'Animation', 'Biography', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir', 'History', 'Horror', 'Music', 'Musical', 'Mystery', 'News', 'Romance', 'Sci-Fi', 'Short', 'Sport', 'Thriller', 'War', 'Western']
Submission shape: (3383, 25)
Sample shape:     (3383, 25)


In [50]:
submission

,ID,p_Action,p_Adventure,p_Animation,p_Biography,p_Comedy,p_Crime,p_Documentary,p_Drama,p_Family,...,p_Musical,p_Mystery,p_News,p_Romance,p_Sci-Fi,p_Short,p_Sport,p_Thriller,p_War,p_Western
0,1,0,0,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,0,0
2,4,0,0,0,1,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,0,0,0,0,0,1,0,1,0,...,0,1,0,0,0,0,0,1,0,0
6,6,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,1,0,0
8,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6758,11263,1,0,0,0,1,1,0,1,0,...,0,0,0,0,0,0,0,1,0,0
6760,11265,0,0,0,0,1,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
6762,11269,0,0,0,0,0,0,0,1,0,...,1,0,0,1,0,0,0,0,0,0
6764,11270,0,1,1,0,1,0,0,1,1,...,1,0,0,0,0,0,0,0,0,0
